# 가상 계측 모델

In [14]:
import numpy as np
import pandas as pd
from statsmodels.stats.outliers_influence import variance_inflation_factor
from sklearn.linear_model import LassoCV
from sklearn.preprocessing import StandardScaler

## 가상 센서 데이터 생성
- 온도
- 압력
- 히터 전류
- 온도 무관 Gas Flow 가정

- 두께 : 온도, Gas Flow에 의해 형성을 가정

In [15]:
np.random.seed(25)
n_wafers = 500

temp = np.random.normal(80, 2, n_wafers)
pressure = temp * 1.5 + np.random.normal(20, 0.5, n_wafers)
heater_current = temp * 0.8 + np.random.normal(5, 0.2, n_wafers)
unrelated_flow = np.random.normal(100, 10, n_wafers)

thickness = temp * 4.0 + unrelated_flow * 0.5 + np.random.normal(0, 2, n_wafers)

X_data = pd.DataFrame({
    "Temp": temp,
    "Pressure": pressure,
    "Heater_Current": heater_current,
    "Gas_Flow": unrelated_flow
})

## 다중공선성 판단
- VIF 연산
    - $VIF=\frac{1}{1-R^2}$
    - VIF > 10 -> 다중 공선성이 높은 변수

In [16]:
st_scaler = StandardScaler()
X_scaled_temp = st_scaler.fit_transform(X_data)
df_scaled = pd.DataFrame(X_scaled_temp, columns=X_data.columns)

vif_data = pd.DataFrame()
vif_data["Feature"] = df_scaled.columns
vif_data["VIF"] = [variance_inflation_factor(df_scaled.values, i) for i in range(df_scaled.shape[1])]

print("--- VIF Analysis (Multicollinearity Check) ---")
print(vif_data)

--- VIF Analysis (Multicollinearity Check) ---
          Feature         VIF
0            Temp  110.491571
1        Pressure   38.480268
2  Heater_Current   74.493491
3        Gas_Flow    1.003323


## Lasso 회귀
- L1 Regularization
    - Temp / Pressure / Heater_Current / Gas_Flow 중 VIF 지수가 높은 Temp, Pressure, Heater_Current 확인
    - 영향력이 가장 높은 Temp (VIF = 110.491571) 변수의 계수만 활성화, 나머지 VIF>10인 Pressure, Heater_Current는 0으로 수축

In [17]:
lasso = LassoCV(cv=5, random_state=42)
lasso.fit(X_scaled_temp, thickness)

print("\n--- Optimized Lasso Coefficients ---")
for name, coef in zip(X_data.columns, lasso.coef_):
    print(f"{name}: {coef:.4f}")


--- Optimized Lasso Coefficients ---
Temp: 7.8488
Pressure: 0.2653
Heater_Current: 0.2838
Gas_Flow: 4.9530
